In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from openai import OpenAI
openai_client = OpenAI()

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini", input=prompt
    )
    return response.output_text

In [8]:
question: str = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Probably yes — it depends on the course’s enrollment policy and whether it’s still open.

A few quick possibilities:
- **Open enrollment:** you can usually join anytime.
- **Fixed cohort:** you may need to wait for the next start date.
- **In progress:** some courses allow late registration for a short period.
- **Prerequisites or approval needed:** you might have to contact the instructor/admin.

If you want, I can help you draft a short message to ask the course team whether you can still enroll.


In [9]:
context: str = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt: str = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}



In [12]:
question: str = "I just discovered the course. Can I join now?"
answer = llm(prompt.format(question=question, context=context))
print(answer)

Yes, you can still join now. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


**Exemplary RAG function:**

```python
def rag(question, context):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    answer = llm(user_prompt)
    return answer
```

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [15]:
documents[1100]

{'id': 'f580e98fdc',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Waitress on Windows / Git Bash: "waitress-serve: command not found"',
 'answer': '`pip install waitress` does install `waitress-serve.exe`, but Python\'s `Scripts/` directory may not be on Git Bash\'s `PATH`. The pip output usually warns about this:\n\n```\nWARNING: The script waitress-serve.exe is installed in \'C:\\Users\\<you>\\...\\Scripts\'\nwhich is not on PATH.\n```\n\nTo fix, add that `Scripts` directory to Git Bash\'s `PATH` permanently:\n\n```bash\nnano ~/.bashrc\n# add this line, with the path from the warning:\nexport PATH="/c/Users/<you>/.../Scripts:$PATH"\n```\n\nClose Git Bash and reopen it. `waitress-serve --help` should now work.\n\nIf you\'re using `uv`, this isn\'t an issue because `uv run waitress-serve ...` runs the binary directly from the venv without needing it on `PATH`.'}

## `search()`

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
)

index.fit(documents)

In [17]:
index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5,
)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [18]:
def search(question, course: str = "llm-zoomcamp") -> list[dict]:
    """
    Search the index for the question.
    
    Parameters
    ----------
    question: str
        The question to search for.
    course: str
        The course to search for.
        
    Returns
    -------
    list[dict]
        The search results.
    """
    boost_dict: dict[str, float] = {"question": 2.0, "section": 0.5}
    filter_dict: dict[str, str] = {"course": course}
    num_results: int = 5
    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results,
    )

In [19]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## `build_promtp()`

In [21]:
INSTRUCTIONS: str = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
"""

In [22]:
USER_PROMPT_TEMPLATE: str = """
Question:
{question}

Context:
{context}
"""

In [23]:
def build_context(search_results: list[dict]) -> str:
    """
    Build the context for the prompt.
    
    Parameters
    ----------
    search_results: list[dict]
        The search results.
        
    Returns
    -------
    str
        The context for the prompt.
    """
    lines = []
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        
    return "\n".join(lines).strip()


In [29]:
def build_prompt(question: str, search_results: list[dict]) -> str:
    """
    Build the prompt for the LLM.

    Parameters
    ----------
    question: str
        The question to answer.
    search_results: list[dict]
        The search results.

    Returns
    -------
    str
        The prompt for the LLM.
    """
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
    )
    return prompt.strip()


prompt = build_prompt(question, search_results)
print("#" * 100)
print("PROMPT:")
print(prompt)
print("#" * 100)
answer = llm(prompt)
print("ANSWER:")
print(answer)
print("#" * 100)


####################################################################################################
PROMPT:
Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates